# AquaRender — One-shot Watercolor (Kaggle)

Single-prompt, single-image end-to-end run on a Kaggle GPU. Edit the **`PROMPT`**
and **`INPUT_IMAGE_URL`** cell below, **Run All**, and the final cell shows your
watercolor inline.

This notebook is self-contained — you don't need the local Streamlit app. Setup
is identical to `aquarender_kaggle.ipynb` (idempotent: re-runs skip already-
downloaded models). The actual generation is one `bash` cell that drives
ComfyUI's HTTP API directly.

**Settings:**
1. Settings → Accelerator → **GPU P100** (or T4)
2. Settings → Internet → **On**

⏱️  Cold first run: ~5–7 min (model downloads). Warm reruns: ~25 s.


## ⚙️ Settings — edit these

In [ ]:
# The prompt anchor: keep "watercolor painting" or "watercolor style" in the
# positive prompt; the LoRA was trained on it.
PROMPT = "watercolor painting, soft brushstrokes, paper texture, gentle color bleed, hand-painted"
NEGATIVE = "photo, photograph, photorealistic, 3d render, oil painting, harsh edges, oversaturated, low quality, blurry"

# Reference image: any URL OR a path under /kaggle/input/<dataset>/...
INPUT_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/640px-PNG_transparency_demonstration_1.png"
# INPUT_IMAGE_PATH = "/kaggle/input/your-dataset/your-image.jpg"  # alternative

# Tuning knobs
STEPS = 28
CFG = 5.5
DENOISE = 0.5          # 0.35 = light wash, 0.65 = strong stylization
LORA_WEIGHT = 0.8      # 0.6–1.0
CN_STRENGTH = 0.85     # 0.5 = loose structure, 0.9 = tight structure
SEED = 0               # 0 = random; any int = reproducible

# Optional: swap in your own LoRA if you've added it as a Kaggle Dataset.
# Cell 6 symlinks /kaggle/input/*/.safetensors into ComfyUI's loras folder.
LORA_NAME = "watercolor_style_lora_sdxl.safetensors"


## 1. GPU check

In [ ]:
import subprocess
try:
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"
    ])
    print("GPU detected:")
    print(out.decode())
except Exception:
    raise SystemExit(
        "❌ No GPU. Open Settings → Accelerator → GPU P100 (or T4) and re-run."
    )


## 2. Clone ComfyUI (pinned, idempotent)

In [ ]:
import subprocess, pathlib
COMFY_DIR = pathlib.Path("/kaggle/working/ComfyUI")
COMFY_COMMIT = "b9d31a9b9b0bff58b85c9e34e62fa7b9b3b5d5c3"

if not COMFY_DIR.exists():
    subprocess.check_call([
        "git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)
    ])
subprocess.check_call(
    ["git", "-C", str(COMFY_DIR), "fetch", "--depth=1", "origin", COMFY_COMMIT],
    stderr=subprocess.DEVNULL,
)
subprocess.check_call(["git", "-C", str(COMFY_DIR), "checkout", COMFY_COMMIT])
subprocess.check_call(["pip", "install", "--quiet", "-r", str(COMFY_DIR / "requirements.txt")])
print("ComfyUI ready at", COMFY_DIR)


## 3. Download models (idempotent: skips if already present)

In [ ]:
import subprocess, pathlib

def fetch(url, dest):
    if dest.exists():
        print(f"  ✓ {dest.name} ({dest.stat().st_size / 1e6:.0f} MB) already on disk")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  ⬇ {dest.name} from {url}")
    subprocess.check_call(["wget", "-q", "--show-progress", "-O", str(dest), url])

ROOT = pathlib.Path("/kaggle/working/ComfyUI/models")
fetch(
    "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
    ROOT / "checkpoints/sd_xl_base_1.0.safetensors",
)
fetch(
    "https://huggingface.co/ostris/watercolor_style_lora_sdxl/resolve/main/watercolor_style_lora_sdxl.safetensors",
    ROOT / "loras/watercolor_style_lora_sdxl.safetensors",
)
fetch(
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_lineart_full.safetensors",
    ROOT / "controlnet/diffusers_xl_lineart_full.safetensors",
)
print("✓ all models in place")


## 4. Symlink any user-uploaded LoRA datasets

In [ ]:
import pathlib, os
input_root = pathlib.Path("/kaggle/input")
lora_dir = pathlib.Path("/kaggle/working/ComfyUI/models/loras")
linked = []
if input_root.exists():
    for ds in input_root.iterdir():
        if not ds.is_dir():
            continue
        for f in ds.rglob("*.safetensors"):
            link = lora_dir / f.name
            if link.exists():
                continue
            try:
                os.symlink(f, link)
                linked.append(f.name)
            except OSError:
                pass
print(f"Linked {len(linked)} LoRA(s) from /kaggle/input/: {linked}")


## 5. Start ComfyUI in the background

In [ ]:
import subprocess, time, pathlib, urllib.request, json, os, signal

LOG = pathlib.Path("/kaggle/working/comfyui.log")

# Kill any prior instance from a re-run.
try:
    subprocess.check_call(["pkill", "-f", "ComfyUI/main.py"], stderr=subprocess.DEVNULL)
    time.sleep(2)
except subprocess.CalledProcessError:
    pass

proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/kaggle/working/ComfyUI",
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)
print("ComfyUI PID:", proc.pid, "(log:", LOG, ")")

deadline = time.time() + 240
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            stats = json.load(r)
            print("✅ ComfyUI ready —", stats.get("system", {}).get("comfyui_version", "?"))
            break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("ComfyUI never became ready. tail /kaggle/working/comfyui.log")


## 6. Fetch the input image

In [ ]:
import urllib.request, pathlib, os

INPUT_PATH = pathlib.Path("/tmp/aquarender_input.png")

if 'INPUT_IMAGE_PATH' in dir() and INPUT_IMAGE_PATH:
    src = pathlib.Path(INPUT_IMAGE_PATH)
    INPUT_PATH.write_bytes(src.read_bytes())
    print(f"copied {src} → {INPUT_PATH}")
else:
    urllib.request.urlretrieve(INPUT_IMAGE_URL, INPUT_PATH)
    print(f"downloaded {INPUT_IMAGE_URL} → {INPUT_PATH}")

from IPython.display import Image, display
display(Image(str(INPUT_PATH), width=400))


## 7. Run the one-shot bash script\n\nWrites `oneshot.sh` to disk, then drives ComfyUI's HTTP API in pure curl + jq — no Python deps.

In [ ]:
# Write the same scripts/oneshot.sh that ships in the repo.
%%writefile /kaggle/working/oneshot.sh
#!/usr/bin/env bash
set -euo pipefail

COMFY_URL="${COMFY_URL:-http://127.0.0.1:8188}"
OUTPUT="${OUTPUT:-/tmp/aquarender_oneshot.png}"
PROMPT="${PROMPT:?PROMPT is required}"
NEGATIVE="${NEGATIVE:-}"
CHECKPOINT="${CHECKPOINT:-sd_xl_base_1.0.safetensors}"
LORA_NAME="${LORA_NAME:-watercolor_style_lora_sdxl.safetensors}"
LORA_WEIGHT="${LORA_WEIGHT:-0.8}"
CN_NAME="${CN_NAME:-diffusers_xl_lineart_full.safetensors}"
CN_STRENGTH="${CN_STRENGTH:-0.85}"
STEPS="${STEPS:-28}"
CFG="${CFG:-5.5}"
DENOISE="${DENOISE:-0.5}"
SAMPLER="${SAMPLER:-dpmpp_2m_sde}"
SCHEDULER="${SCHEDULER:-karras}"
SEED="${SEED:-$((RANDOM * RANDOM))}"
TIMEOUT_S="${TIMEOUT_S:-180}"
INPUT_IMAGE="${INPUT_IMAGE:?INPUT_IMAGE is required}"
CLIENT_ID="aquarender-oneshot-$$"

echo "→ engine: $COMFY_URL"
echo "→ seed:   $SEED"

UPLOAD=$(curl -sf -X POST -F "image=@${INPUT_IMAGE}" -F "type=input" -F "overwrite=true" "${COMFY_URL}/upload/image")
IMAGE_NAME=$(echo "$UPLOAD" | jq -r '.name')
echo "→ uploaded as: $IMAGE_NAME"

WORKFLOW=$(jq -n \
  --arg ckpt "$CHECKPOINT" --arg lora "$LORA_NAME" --argjson lora_w "$LORA_WEIGHT" \
  --arg cn "$CN_NAME" --argjson cn_s "$CN_STRENGTH" \
  --arg image "$IMAGE_NAME" --arg pos "$PROMPT" --arg neg "$NEGATIVE" \
  --argjson seed "$SEED" --argjson steps "$STEPS" --argjson cfg "$CFG" --argjson denoise "$DENOISE" \
  --arg sampler "$SAMPLER" --arg scheduler "$SCHEDULER" --arg client_id "$CLIENT_ID" \
  '{
    "prompt": {
      "1": {"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":$ckpt}},
      "2": {"class_type":"LoraLoader","inputs":{"lora_name":$lora,"strength_model":$lora_w,"strength_clip":$lora_w,"model":["1",0],"clip":["1",1]}},
      "3": {"class_type":"CLIPTextEncode","inputs":{"text":$pos,"clip":["2",1]}},
      "4": {"class_type":"CLIPTextEncode","inputs":{"text":$neg,"clip":["2",1]}},
      "5": {"class_type":"LoadImage","inputs":{"image":$image}},
      "6": {"class_type":"VAEEncode","inputs":{"pixels":["5",0],"vae":["1",2]}},
      "7": {"class_type":"ControlNetLoader","inputs":{"control_net_name":$cn}},
      "10": {"class_type":"ControlNetApplyAdvanced","inputs":{"positive":["3",0],"negative":["4",0],"control_net":["7",0],"image":["5",0],"strength":$cn_s,"start_percent":0.0,"end_percent":1.0}},
      "8": {"class_type":"KSampler","inputs":{"model":["2",0],"positive":["10",0],"negative":["10",1],"latent_image":["6",0],"seed":$seed,"steps":$steps,"cfg":$cfg,"sampler_name":$sampler,"scheduler":$scheduler,"denoise":$denoise}},
      "11": {"class_type":"VAEDecode","inputs":{"samples":["8",0],"vae":["1",2]}},
      "9": {"class_type":"SaveImage","inputs":{"images":["11",0],"filename_prefix":"aquarender_oneshot"}}
    },
    "client_id": $client_id
  }')

PROMPT_ID=$(curl -sf -X POST -H 'Content-Type: application/json' -d "$WORKFLOW" "${COMFY_URL}/prompt" | jq -r '.prompt_id')
echo "→ prompt_id: $PROMPT_ID"

DEADLINE=$(( $(date +%s) + TIMEOUT_S ))
while :; do
  if (( $(date +%s) > DEADLINE )); then
    echo "✗ generation exceeded ${TIMEOUT_S}s" >&2
    exit 3
  fi
  HIST=$(curl -sf "${COMFY_URL}/history/${PROMPT_ID}" || true)
  COMPLETED=$(echo "$HIST" | jq -r --arg id "$PROMPT_ID" '.[$id].status.completed // false')
  if [[ "$COMPLETED" == "true" ]]; then break; fi
  sleep 1
done

OUT_FN=$(echo "$HIST" | jq -r --arg id "$PROMPT_ID" '.[$id].outputs."9".images[0].filename')
OUT_SF=$(echo "$HIST" | jq -r --arg id "$PROMPT_ID" '.[$id].outputs."9".images[0].subfolder // ""')
OUT_TY=$(echo "$HIST" | jq -r --arg id "$PROMPT_ID" '.[$id].outputs."9".images[0].type // "output"')

curl -sfG "${COMFY_URL}/view" \
  --data-urlencode "filename=${OUT_FN}" \
  --data-urlencode "subfolder=${OUT_SF}" \
  --data-urlencode "type=${OUT_TY}" \
  -o "$OUTPUT"

echo "→ saved → $OUTPUT (seed=$SEED)"


In [ ]:
import subprocess, os, time, pathlib

OUTPUT_PATH = pathlib.Path("/tmp/aquarender_oneshot.png")

env = {
    **os.environ,
    "INPUT_IMAGE": str(INPUT_PATH),
    "OUTPUT": str(OUTPUT_PATH),
    "PROMPT": PROMPT,
    "NEGATIVE": NEGATIVE,
    "LORA_NAME": LORA_NAME,
    "LORA_WEIGHT": str(LORA_WEIGHT),
    "CN_STRENGTH": str(CN_STRENGTH),
    "STEPS": str(STEPS),
    "CFG": str(CFG),
    "DENOISE": str(DENOISE),
}
if SEED:
    env["SEED"] = str(SEED)

start = time.time()
subprocess.check_call(["bash", "/kaggle/working/oneshot.sh"], env=env)
print(f"⏱  total wall time: {time.time() - start:.1f}s")


## 8. Show the result

In [ ]:
from IPython.display import Image, display, Markdown
import pathlib

display(Markdown("### Input"))
display(Image("/tmp/aquarender_input.png", width=512))

display(Markdown("### Watercolor"))
display(Image("/tmp/aquarender_oneshot.png", width=512))

print("Output:", pathlib.Path("/tmp/aquarender_oneshot.png").stat().st_size, "bytes")


## What now?

- **Re-run cell 7** with a new `PROMPT` / `SEED` / `CN_STRENGTH` to iterate.
- For batches and a real UI, run the full `aquarender_kaggle.ipynb` instead — that one exposes a tunnel URL you paste into the local Streamlit app.
- All outputs land at `/tmp/aquarender_oneshot.png` and are overwritten on each run; download from the Kaggle file browser before the session ends.
